In [7]:
# Import shared loading/cleaning functions from utils.py
from utils import load_clean_data
from utils import get_cv_folds

import pandas as pd
import numpy as np

# category_encoders provides the CatBoostEncoder implementation
# (an "ordered" variant of target encoding, designed to reduce leakage
# and instability compared to plain target encoding)
import category_encoders as ce

# Models we will train and compare
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor

# For scaling numeric features (needed for Linear Regression and KNN)
from sklearn.preprocessing import StandardScaler

# For evaluation metrics
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score

In [8]:
# Load the cleaned dataset
df = load_clean_data("data.csv")

# Generate the FIXED set of cross-validation folds
folds = get_cv_folds(df)

print(f"Loaded {df.shape[0]} rows, {df.shape[1]} columns")
print(f"Generated {len(folds)} folds")

Loaded 79159 rows, 17 columns
Generated 5 folds


In [9]:
# The target variable we're predicting
y = df["advertised_price"]

# Drop the target from the feature set
X = df.drop(columns=["advertised_price"])

# Categorical columns to encode
categorical_cols = ["make", "model", "generation", "fuel", "body",
                     "transmission", "trim", "plate_age_id"]

# Numeric columns — vehicle_age_days excluded (redundant with vehicle_age_years)
numeric_cols = ["mileage", "list_price", "reg_year", "doors", "engine",
                "engine_missing", "vehicle_age_years"]

print(f"Categorical columns: {len(categorical_cols)}")
print(f"Numeric columns: {len(numeric_cols)}")

Categorical columns: 8
Numeric columns: 7


In [10]:
# Reset results dictionary — storing RMSE, MAE, and R² for each model, per fold
results = {
    "Linear Regression": {"rmse": [], "mae": [], "r2": []},
    "KNN": {"rmse": [], "mae": [], "r2": []},
    "Decision Tree": {"rmse": [], "mae": [], "r2": []},
    "Random Forest": {"rmse": [], "mae": [], "r2": []},
    "Stacking": {"rmse": [], "mae": [], "r2": []}
}

In [11]:
# Loop through each of the 5 fixed folds
for fold_num, (train_idx, test_idx) in enumerate(folds, start=1):

    # ---- Split data into this fold's train/test portions ----
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    # ---- CatBoost Encode the categorical columns ----
    # cols=categorical_cols tells the encoder which columns to transform,
    # leaving numeric columns untouched if they were passed in too.
    # random_state ensures the internal ordering/noise process is reproducible.
    cbe = ce.CatBoostEncoder(cols=categorical_cols, random_state=42)

    # IMPORTANT: fit using BOTH X_train and y_train, since CatBoost Encoding
    # (like Target Encoding) needs the target values to compute its statistics.
    # Fitting only on the training fold prevents test data from leaking in.
    cbe.fit(X_train[categorical_cols], y_train)

    # Transform both train and test using the SAME fitted encoder
    X_train_cat = cbe.transform(X_train[categorical_cols])
    X_test_cat = cbe.transform(X_test[categorical_cols])

    # ---- Combine encoded categorical columns with numeric columns ----
    X_train_final = pd.concat([X_train[numeric_cols], X_train_cat], axis=1)
    X_test_final = pd.concat([X_test[numeric_cols], X_test_cat], axis=1)

    # ---- Scale features for models that need it (Linear Regression, KNN) ----
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_final)
    X_test_scaled = scaler.transform(X_test_final)

    # ---- Define the 5 models ----
    linear_model = LinearRegression()
    knn_model = KNeighborsRegressor()
    tree_model = DecisionTreeRegressor(random_state=42)
    rf_model = RandomForestRegressor(random_state=42, n_estimators=50)

    stacking_model = StackingRegressor(
        estimators=[
            ("linear", LinearRegression()),
            ("knn", KNeighborsRegressor()),
            ("tree", DecisionTreeRegressor(random_state=42)),
            ("rf", RandomForestRegressor(random_state=42, n_estimators=50))
        ],
        final_estimator=LinearRegression(),
        cv=3,
    )

    # ---- Train and evaluate each model ----

    # Linear Regression (scaled features)
    linear_model.fit(X_train_scaled, y_train)
    linear_preds = linear_model.predict(X_test_scaled)
    results["Linear Regression"]["rmse"].append(root_mean_squared_error(y_test, linear_preds))
    results["Linear Regression"]["mae"].append(mean_absolute_error(y_test, linear_preds))
    results["Linear Regression"]["r2"].append(r2_score(y_test, linear_preds))

    # KNN (scaled features)
    knn_model.fit(X_train_scaled, y_train)
    knn_preds = knn_model.predict(X_test_scaled)
    results["KNN"]["rmse"].append(root_mean_squared_error(y_test, knn_preds))
    results["KNN"]["mae"].append(mean_absolute_error(y_test, knn_preds))
    results["KNN"]["r2"].append(r2_score(y_test, knn_preds))

    # Decision Tree (unscaled features)
    tree_model.fit(X_train_final, y_train)
    tree_preds = tree_model.predict(X_test_final)
    results["Decision Tree"]["rmse"].append(root_mean_squared_error(y_test, tree_preds))
    results["Decision Tree"]["mae"].append(mean_absolute_error(y_test, tree_preds))
    results["Decision Tree"]["r2"].append(r2_score(y_test, tree_preds))

    # Random Forest (unscaled features)
    rf_model.fit(X_train_final, y_train)
    rf_preds = rf_model.predict(X_test_final)
    results["Random Forest"]["rmse"].append(root_mean_squared_error(y_test, rf_preds))
    results["Random Forest"]["mae"].append(mean_absolute_error(y_test, rf_preds))
    results["Random Forest"]["r2"].append(r2_score(y_test, rf_preds))

    # Stacking (unscaled features)
    stacking_model.fit(X_train_final, y_train)
    stacking_preds = stacking_model.predict(X_test_final)
    results["Stacking"]["rmse"].append(root_mean_squared_error(y_test, stacking_preds))
    results["Stacking"]["mae"].append(mean_absolute_error(y_test, stacking_preds))
    results["Stacking"]["r2"].append(r2_score(y_test, stacking_preds))

    print(f"Fold {fold_num}/{len(folds)} complete")

print("\nAll folds complete.")

Fold 1/5 complete
Fold 2/5 complete
Fold 3/5 complete
Fold 4/5 complete
Fold 5/5 complete

All folds complete.


In [12]:
# Build a summary table: mean and std of RMSE, MAE, and R² across the folds, per model
summary_rows = []
for model_name, metrics in results.items():
    summary_rows.append({
        "model": model_name,
        "mean_rmse": np.mean(metrics["rmse"]),
        "std_rmse": np.std(metrics["rmse"]),
        "mean_mae": np.mean(metrics["mae"]),
        "std_mae": np.std(metrics["mae"]),
        "mean_r2": np.mean(metrics["r2"]),
        "std_r2": np.std(metrics["r2"])
    })

summary = pd.DataFrame(summary_rows)
summary

,model,mean_rmse,std_rmse,mean_mae,std_mae,mean_r2,std_r2
0,Linear Regression,4216.367866,114.171461,2630.148998,34.083928,0.910075,0.003269
1,KNN,2353.538551,133.803687,1302.470901,28.530849,0.971959,0.002531
2,Decision Tree,2789.951415,209.420752,1529.689406,28.270346,0.960537,0.004977
3,Random Forest,1983.797686,114.443872,1083.112886,13.218168,0.980080,0.001772
4,Stacking,1977.490811,120.506396,1094.491550,15.851924,0.980202,0.001893


In [13]:
# Flatten results into a long-format DataFrame: one row per model per fold per metric
rows = []
for model_name, metrics in results.items():
    for fold_i in range(len(folds)):
        rows.append({
            "encoder": "catboost",
            "model": model_name,
            "fold": fold_i + 1,
            "rmse": metrics["rmse"][fold_i],
            "mae": metrics["mae"][fold_i],
            "r2": metrics["r2"][fold_i]
        })

results_df = pd.DataFrame(rows)
results_df.to_csv("catboost_scores.csv", index=False)

print("Saved results to catboost_scores.csv")

Saved results to catboost_scores.csv


In [14]:
# Capture feature importance from Random Forest, same as previous notebooks
feature_importance = pd.DataFrame({
    "feature": X_train_final.columns,
    "importance": rf_model.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance["encoder"] = "catboost"
feature_importance.to_csv("catboost_feature_importance.csv", index=False)

feature_importance.head(15)

,feature,importance,encoder
1,list_price,0.582381,catboost
9,generation,0.177488,catboost
0,mileage,0.104118,catboost
8,model,0.052709,catboost
6,vehicle_age_years,0.023818,catboost
13,trim,0.018179,catboost
14,plate_age_id,0.016863,catboost
2,reg_year,0.009334,catboost
7,make,0.005586,catboost
4,engine,0.003854,catboost
